# Notebook 8b: Is it the polish, or just the cleaning? (control for RQ5)

**Before running:** Runtime → Change runtime type → **T4 GPU**. Notebook 8 must have run first (it reads its rewrites from Drive).

**Why this notebook.** In Notebook 8, E5 gave AI-polished resumes a large boost (+0.74 SD). But the original resumes are scraped text with words run together (`SkillsGAAP AccountingAccount Reconciliation`, `03/2017toCurrentAccountant`). The rewrite fixes that too. So part of the boost may come from **clean text**, not from **polished wording**. A reviewer will ask.

**What this notebook does** (the same 181 resumes as Notebook 8)
1. **Clean** each original with simple fixed rules: put a space where two words were glued together, and nothing else. No model, no rewording. A check proves the cleaned text has exactly the same letters and digits as the original.
2. **Plain polished:** the polished text with the markdown (`**`, `*` bullets, `#` headings) stripped, to see whether formatting symbols matter.
3. Scores four versions of each resume with TF-IDF, E5, BGE, MPNet: **original → cleaned → polished**, plus plain polished. The Qwen3-4B judge also scores the cleaned version (original and polished scores are reused from Notebook 8's cache).
4. Splits the total boost into **cleaning** (original → cleaned) and **polish beyond cleaning** (cleaned → polished).
5. Bonus: cleans **every** resume and checks whether ranking quality (NDCG@10) changes. If it improves, cleaning is a preprocessing recommendation for the paper.

**Run time:** about 25–35 minutes (embeddings about 10 min, judge about 8 min). Resumable: the judge saves to Drive as it goes.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip -q install -U transformers accelerate sentence-transformers

import os, re, gc, time
import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon, spearmanr

assert torch.cuda.is_available(), 'No GPU: Runtime > Change runtime type > T4 GPU, then run again'
DATA = '/content/drive/MyDrive/resume-job-matching/data'
RESULTS = '/content/drive/MyDrive/resume-job-matching/results'
CACHE = f'{RESULTS}/llm_cache'
SEED = 42
MAX_WORDS = 900   # the same resume cut as Notebook 8

all_df = pd.concat([pd.read_parquet(f'{DATA}/{s}.parquet').assign(split=s) for s in ('train', 'val', 'test')],
                   ignore_index=True)
g = all_df.groupby('job_id').agg(candidates=('resume_id', 'size'),
                                 relevant=('relevance', lambda r: int((r > 0).sum())),
                                 levels=('relevance', 'nunique'))
usable = g[(g.candidates >= 5) & (g.relevant >= 1) & (g.levels >= 2)].index
evald = all_df[all_df.job_id.isin(usable)].reset_index(drop=True)
evald['pair_key'] = evald.job_id + '|' + evald.resume_id

def cut(text, n_words):
    w = text.split()
    return text if len(w) <= n_words else ' '.join(w[:n_words])
def numbers(t):
    return set(re.findall(r'\d+(?:\.\d+)?', t.replace(',', '')))

# The rewrites from Notebook 8, with the same usable / strict rules
MAX_NEW = 1600
rw = pd.read_parquet(f'{RESULTS}/08_rewrites.parquet')
res_text = evald.drop_duplicates('resume_id').set_index('resume_id').resume_text
rw['original'] = rw.resume_id.map(lambda r: cut(res_text[r], MAX_WORDS))
rw['len_ratio'] = rw.polished.str.split().str.len() / rw.original.str.split().str.len()
rw['usable'] = rw.len_ratio.between(0.6, 1.8) & ~rw.hit_limit & (rw.polished.str.len() > 0)
rw['strict'] = rw.usable & [len(numbers(p) - numbers(o)) == 0 for o, p in zip(rw.original, rw.polished)]
rw = rw[rw.usable].reset_index(drop=True)
print(f'{len(usable)} usable jobs, {len(evald)} pairs')
print(f'{len(rw)} usable rewrites (Notebook 8: 181), {rw.strict.sum()} strict (Notebook 8: 151)')
assert len(rw) == 181 and rw.strict.sum() == 151, 'Not the same rewrites as Notebook 8.'

## 1. Clean the originals, and strip markdown from the polished text

**Cleaning rules** (only spaces are added, nothing else changes):
- Two words glued at a case change: `SummaryMotivated` → `Summary Motivated`, `GAAPAccounting` → `GAAP Accounting`. Real names written that way (`JavaScript`, `PowerPoint`, `QuickBooks`) are kept: a glued word is left alone if it appears in **2 or more job descriptions**, which are cleaner text written by employers.
- A word glued to a number: `Experience03/2017` → `Experience 03/2017`, `2017toCurrent` → `2017 to Current`.
- Punctuation with no space after it: `managers.Applied` → `managers. Applied`, `Morganton,NC` → `Morganton, NC`, and spaces around `|`.

**Messiness** of a resume = spaces added per 100 words. Later we check whether messier resumes gained more from polishing.

In [ ]:
JOINED = re.compile(r'[a-z][A-Z]|[A-Z]{2}[a-z]{2}')   # 2+ lowercase after, so IDs / APIs stay whole
jobs_u = evald.drop_duplicates('job_id')
job_df = {}
for t in jobs_u.job_text:
    for w in set(re.findall(r'[A-Za-z]+', t)):
        if JOINED.search(w):
            job_df[w] = job_df.get(w, 0) + 1
KEEP = {w for w, n in job_df.items() if n >= 2}
print(f'{len(KEEP)} glued-case words kept as real names, e.g.', sorted(KEEP, key=lambda w: -job_df[w])[:20])

split_log = {}
def split_word(w):
    if w in KEEP or not JOINED.search(w):
        return w
    s = re.sub(r'([a-z])([A-Z])', r'\1 \2', w)
    s = re.sub(r'([A-Z]+)([A-Z][a-z]{2,})', r'\1 \2', s)
    split_log[w] = split_log.get(w, 0) + 1
    return s

def clean(t):
    t = re.sub(r'[A-Za-z]+', lambda m: split_word(m.group()), t)
    t = re.sub(r'(?<=[a-z]{4})(?=\d)', ' ', t)                 # Experience03/2017
    t = re.sub(r'(?<=\d)(?=to[A-Z ])|(?<=\d)(?=[A-Z][a-z])', ' ', t)   # 2017toCurrent, 2017Accountant
    t = re.sub(r'(?<=[a-z]{2}[.;:!?])(?=[A-Z])', ' ', t)        # managers.Applied
    t = re.sub(r'(?<=[A-Za-z]{2}[,;])(?=[A-Za-z])', ' ', t)     # Morganton,NC
    t = re.sub(r'\s*\|\s*', ' | ', t)
    return re.sub(r'[ \t]+', ' ', t).strip()

def plain(t):
    t = re.sub(r'^\s*#+\s*', '', t, flags=re.M)                 # headings
    t = re.sub(r'^\s*[-*•]\s+', '', t, flags=re.M)               # bullets
    t = t.replace('**', '').replace('__', '').replace('`', '')
    return re.sub(r'[ \t]+', ' ', t).strip()

evald['resume_cut'] = evald.resume_text.map(lambda t: cut(t, MAX_WORDS))
cleaned = {r: clean(t) for r, t in evald.drop_duplicates('resume_id')[['resume_id', 'resume_cut']].itertuples(index=False)}
evald['resume_clean'] = evald.resume_id.map(cleaned)

rw['clean'] = rw.resume_id.map(cleaned)
rw['plain'] = rw.polished.map(plain)
same = lambda a, b: re.sub(r'\W', '', a) == re.sub(r'\W', '', b)
bad = [r for r, t in cleaned.items() if not same(t, cut(res_text[r], MAX_WORDS))]
assert not bad, f'Cleaning changed the content of {len(bad)} resumes.'
print('check passed: every cleaned resume has exactly the same letters and digits as its original')

all_words = pd.Series({r: len(cut(res_text[r], MAX_WORDS).split()) for r in cleaned})
added = pd.Series({r: len(t.split()) for r, t in cleaned.items()}) - all_words
messy = (100 * added / all_words)
rw['messiness'] = rw.resume_id.map(messy)
print('spaces added per 100 words, all resumes:', messy.describe().round(1).to_dict())
print('most frequent splits:', sorted(split_log.items(), key=lambda kv: -kv[1])[:25])

ex = rw.iloc[0]
print('\n===== ORIGINAL (first 80 words) =====\n', ' '.join(ex.original.split()[:80]))
print('\n===== CLEANED (first 80 words) =====\n', ' '.join(ex.clean.split()[:80]))
print('\n===== PLAIN POLISHED (first 80 words) =====\n', ' '.join(ex.plain.split()[:80]))

## 2. Score every version: TF-IDF and embeddings

As in Notebook 8, every other candidate of the job keeps its **original** resume; only the sampled resume is swapped. Versions: `orig`, `clean`, `plain` (polished, no markdown), `pol` (polished).

We also score **all** resumes cleaned (`allclean`), for the bonus test in section 5.

**Sanity check:** the `orig` and `pol` scores must match Notebook 8 (tiny GPU differences are fine).

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

ids = set(rw.resume_id)
target = evald[evald.resume_id.isin(ids)].copy()
VERSIONS = ['clean', 'plain', 'pol']
text_of = {v: rw.set_index('resume_id')[{'clean': 'clean', 'plain': 'plain', 'pol': 'polished'}[v]] for v in VERSIONS}
print(f'{len(target)} pairs to score ({target.resume_id.nunique()} resumes, {target.job_id.nunique()} jobs)')

tfidf = TfidfVectorizer(stop_words='english', ngram_range=(1, 2), min_df=2, sublinear_tf=True, max_features=100_000)
tfidf.fit(list(evald.job_text.unique()) + list(evald.resume_cut.unique()))   # original corpus only, as in Notebook 8
def tfidf_cos(jobs, resumes):
    return np.asarray(tfidf.transform(jobs).multiply(tfidf.transform(resumes)).sum(axis=1)).ravel()
evald['tfidf'] = tfidf_cos(evald.job_text, evald.resume_cut)
evald['tfidf_allclean'] = tfidf_cos(evald.job_text, evald.resume_clean)
for v in VERSIONS:
    target['tfidf_' + v] = tfidf_cos(target.job_text, target.resume_id.map(text_of[v]))

MODELS = [('e5', 'intfloat/e5-base-v2', 'query: ', 'passage: '),
          ('bge', 'BAAI/bge-base-en-v1.5', '', ''),
          ('mpnet', 'sentence-transformers/all-mpnet-base-v2', '', '')]
res_u = evald.drop_duplicates('resume_id')
for col, hf_id, qp, dp in MODELS:
    m = SentenceTransformer(hf_id, device='cuda'); m.max_seq_length = 512
    enc = lambda texts, p: m.encode([p + t for t in texts], batch_size=32, normalize_embeddings=True, show_progress_bar=False)
    je = dict(zip(jobs_u.job_id, enc(jobs_u.job_text.tolist(), qp)))
    ro = dict(zip(res_u.resume_id, enc(res_u.resume_cut.tolist(), dp)))
    rc = dict(zip(res_u.resume_id, enc(res_u.resume_clean.tolist(), dp)))
    evald[col] = [float(je[j] @ ro[r]) for j, r in zip(evald.job_id, evald.resume_id)]
    evald[col + '_allclean'] = [float(je[j] @ rc[r]) for j, r in zip(evald.job_id, evald.resume_id)]
    for v in VERSIONS:
        rv = rc if v == 'clean' else dict(zip(rw.resume_id, enc(text_of[v][rw.resume_id].tolist(), dp)))
        target[f'{col}_{v}'] = [float(je[j] @ rv[r]) for j, r in zip(target.job_id, target.resume_id)]
    del m; gc.collect(); torch.cuda.empty_cache()
    print(col, 'done')

CONTENT = ['tfidf', 'e5', 'bge', 'mpnet']
for col in CONTENT:
    target[col + '_orig'] = target.pair_key.map(evald.set_index('pair_key')[col])
    by_job = evald.groupby('job_id')[col].apply(np.array).to_dict()
    # rank within the job (1 = top); the job's list holds this resume's original score, so remove it once
    target[col + '_rank_orig'] = [1 + int((by_job[j] > s).sum()) for j, s in zip(target.job_id, target[col + '_orig'])]
    for v in VERSIONS:
        target[f'{col}_rank_{v}'] = [1 + int((by_job[j] > s).sum()) - int(o > s)
                                     for j, o, s in zip(target.job_id, target[col + '_orig'], target[f'{col}_{v}'])]

nb8 = pd.read_parquet(f'{RESULTS}/08_pair_scores.parquet').set_index('pair_key')
for col in CONTENT:
    for v in ('orig', 'pol'):
        diff = (target.set_index('pair_key')[f'{col}_{v}'] - nb8[f'{col}_{v}']).abs().max()
        print(f'{col:6} {v:5} max difference from Notebook 8: {diff:.5f}')
print('scored (differences above 0.01 mean something changed)')

## 3. The Qwen3-4B judge on the cleaned version

The same pairs as Notebook 8 (one fit and one no-fit per resume). The original and polished scores come from Notebook 8's cache; only the cleaned version is new (about 320 calls).

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

SYSTEM = ('You are an experienced technical recruiter. You judge how well a candidate fits one '
          'specific job, based only on the job description and the resume.')
TEMPLATE = '''Job description:
"""
{job}
"""

Candidate resume:
"""
{resume}
"""

How well does this candidate fit THIS job?
A) Good Fit - meets the main requirements (role, key skills, experience level)
B) Potential Fit - related background, but missing some key requirements
C) No Fit - different field or clearly unqualified

Answer with a single letter: A, B or C.'''

def cut_dots(text, n_words):
    w = text.split()
    return text if len(w) <= n_words else ' '.join(w[:n_words]) + ' ...'

def build_prompt(tok, job, resume):
    msgs = [{'role': 'system', 'content': SYSTEM},
            {'role': 'user', 'content': TEMPLATE.format(job=cut_dots(job, 450), resume=cut_dots(resume, 900))}]
    kwargs = {'tokenize': False, 'add_generation_prompt': True}
    try:
        return tok.apply_chat_template(msgs, enable_thinking=False, **kwargs)
    except TypeError:
        return tok.apply_chat_template(msgs, **kwargs)

def load_judge(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    tok.padding_side = 'left'
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, device_map='cuda')
    model.eval()
    letter_ids = [tok.encode(x, add_special_tokens=False)[0] for x in ('A', 'B', 'C')]
    assert len(set(letter_ids)) == 3
    return tok, model, letter_ids

@torch.no_grad()
def score_pairs(model_id, tag, pairs, batch_size=4, save_every=100):
    path = f'{CACHE}/{tag}.parquet'
    empty = pd.DataFrame({'pair_key': pd.Series(dtype=str), **{c: pd.Series(dtype=float) for c in ('pA', 'pB', 'pC', 'ms')}})
    done = pd.read_parquet(path) if os.path.exists(path) else empty
    todo = pairs[~pairs.pair_key.isin(set(done.pair_key))]
    print(f'{tag}: {len(done)} cached, {len(todo)} to score')
    if len(todo) == 0:
        return done
    tok, model, letter_ids = load_judge(model_id)
    new_rows = []
    for start in range(0, len(todo), batch_size):
        batch = todo.iloc[start:start + batch_size]
        prompts = [build_prompt(tok, j, r) for j, r in zip(batch.job_text, batch.resume_text)]
        enc = tok(prompts, return_tensors='pt', padding=True).to('cuda')
        torch.cuda.synchronize(); t0 = time.perf_counter()
        logits = model(**enc).logits[:, -1, :].float()
        torch.cuda.synchronize(); ms = (time.perf_counter() - t0) * 1000 / len(batch)
        probs = torch.softmax(logits[:, letter_ids], dim=-1).cpu().numpy()
        if np.isnan(probs).any():
            raise RuntimeError('NaN in model output (fp16 overflow).')
        for key, p in zip(batch.pair_key, probs):
            new_rows.append({'pair_key': key, 'pA': p[0], 'pB': p[1], 'pC': p[2], 'ms': ms})
        if len(new_rows) >= save_every or start + batch_size >= len(todo):
            done = pd.concat([d for d in (done, pd.DataFrame(new_rows)) if len(d)], ignore_index=True)
            done.to_parquet(path, index=False)
            new_rows = []
            print(f'  {len(done)}/{len(pairs)} pairs scored')
    del model; gc.collect(); torch.cuda.empty_cache()
    return done

# The same picks as Notebook 8 (same seed, same order)
pick_rng = np.random.default_rng(SEED)
picked = []
for rid, grp in target.groupby('resume_id', sort=True):
    for want_fit in (True, False):
        c = grp[(grp.relevance > 0) == want_fit]
        if len(c):
            picked.append(c.iloc[int(pick_rng.integers(len(c)))])
J = pd.DataFrame(picked).reset_index(drop=True)
nb8_J = pd.read_parquet(f'{RESULTS}/08_judge_scores.parquet')
assert list(J.pair_key) == list(nb8_J.pair_key), 'Judge pairs differ from Notebook 8.'

old = pd.read_parquet(f'{CACHE}/nb8-qwen3-4b-polish.parquet').set_index('pair_key')
new = score_pairs('Qwen/Qwen3-4B-Instruct-2507', 'nb8b-qwen3-4b-clean',
                  J.assign(pair_key=J.pair_key + '|clean', resume_text=J.resume_clean)).set_index('pair_key')
LABELS = np.array(['Good Fit', 'Potential Fit', 'No Fit'])
for v, src in (('orig', old), ('pol', old), ('clean', new)):
    s = src.loc[J.pair_key + '|' + v]
    J['qwen4b_' + v] = (2 * s.pA + s.pB).to_numpy()
    J['qwen4b_label_' + v] = LABELS[s[['pA', 'pB', 'pC']].to_numpy(dtype=float).argmax(axis=1)]
print('judged')

## 4. Results: how much of the boost is cleaning?

Each row is one step between two versions of the same resumes:
- **cleaning**: original → cleaned (only spaces added)
- **polish beyond cleaning**: cleaned → polished (the real "AI polish" effect)
- **markdown symbols**: plain polished → polished
- **total**: original → polished (should match Notebook 8)

Change is in **standard deviations** of that method's original scores (comparable across methods), with a 95% bootstrap CI over resumes; Wilcoxon p with Holm correction across methods within each step. **Share of total** = cleaning ÷ total.

**How to read it.** If *polish beyond cleaning* for E5 is still clearly positive, "embeddings reward polished wording" stands. If almost all of the total is *cleaning*, the finding becomes "embeddings are sensitive to text formatting", and cleaning is a fix.

In [ ]:
def holm(p):
    p = np.asarray(p, dtype=float); order = np.argsort(p); k = len(p); adj = np.empty(k); running = 0
    for rank, i in enumerate(order):
        running = max(running, (k - rank) * p[i]); adj[i] = min(1, running)
    return adj

def cluster_ci(d, groups, n=2000):
    df = pd.DataFrame({'d': d, 'g': groups}); sums = df.groupby('g').d.agg(['sum', 'count'])
    rng = np.random.default_rng(SEED); ids_ = np.arange(len(sums)); out = []
    for _ in range(n):
        b = sums.iloc[rng.choice(ids_, len(ids_))]
        out.append(b['sum'].sum() / b['count'].sum())
    return np.percentile(out, [2.5, 97.5])

STEPS = [('cleaning', 'orig', 'clean'), ('polish beyond cleaning', 'clean', 'pol'),
         ('markdown symbols', 'plain', 'pol'), ('total', 'orig', 'pol')]

def step_table(keep_ids, mask_fn=None, label='all pairs'):
    rows = []
    sources = [(c, target, evald[c].std()) for c in CONTENT] + [('qwen4b judge', J, J.qwen4b_orig.std())]
    for step, a, b in STEPS:
        for name, df, sd in sources:
            col_a, col_b = (f'{name}_{a}', f'{name}_{b}') if name in CONTENT else (f'qwen4b_{a}', f'qwen4b_{b}')
            if col_a not in df or col_b not in df:
                continue
            x = df[df.resume_id.isin(keep_ids)]
            if mask_fn is not None:
                x = x[mask_fn(x)]
            d = (x[col_b] - x[col_a]).to_numpy()
            lo, hi = cluster_ci(d, x.resume_id.to_numpy())
            row = {'step': step, 'method': name, 'pairs': label, 'n': len(d), 'change in SDs': round(d.mean() / sd, 3),
                   'CI (SDs)': f'[{lo / sd:.3f}, {hi / sd:.3f}]', 'share up': round((d > 0).mean(), 3),
                   'p': wilcoxon(d).pvalue if np.any(d != 0) else 1.0}
            if name in CONTENT:
                row['mean places up'] = round((x[f'{name}_rank_{a}'] - x[f'{name}_rank_{b}']).mean(), 2)
            rows.append(row)
    t = pd.DataFrame(rows)
    t['p (Holm)'] = t.groupby('step').p.transform(lambda s: holm(s.to_numpy()))
    return t

pd.set_option('display.width', 250)
main = step_table(set(rw.resume_id))
strict = step_table(set(rw.resume_id[rw.strict]))
share = main.pivot(index='method', columns='step', values='change in SDs')
share['cleaning share of total'] = (share['cleaning'] / share['total']).round(2)
print(share[['cleaning', 'polish beyond cleaning', 'markdown symbols', 'total', 'cleaning share of total']], '\n')
main

In [ ]:
# Sensitivity: rewrites with no new numbers (151 resumes). The conclusion should not change.
strict

In [ ]:
# (a) Is the polish-beyond-cleaning boost the same for fit and no-fit pairs? (per resume: fit change minus no-fit change)
for name in CONTENT:
    d = target.assign(d=target[f'{name}_pol'] - target[f'{name}_clean'], fit=target.relevance > 0)
    per = d.groupby(['resume_id', 'fit']).d.mean().unstack().dropna()
    gap = (per[True] - per[False]) / evald[name].std()
    p = wilcoxon(gap).pvalue if len(gap) >= 10 else np.nan
    print(f'{name:6} polish beyond cleaning, fit minus no-fit: {gap.mean():+.3f} SDs over {len(gap)} resumes (p = {p:.3g})')

# (b) Do messier resumes gain more? (per resume, Spearman)
print()
for name in CONTENT:
    per = target.assign(c=target[f'{name}_clean'] - target[f'{name}_orig'],
                        t=target[f'{name}_pol'] - target[f'{name}_orig']).groupby('resume_id')[['c', 't']].mean()
    m = rw.set_index('resume_id').messiness[per.index]
    rc, pc = spearmanr(m, per.c); rt, pt = spearmanr(m, per.t)
    print(f'{name:6} messiness vs cleaning gain: rho = {rc:+.2f} (p = {pc:.2g}) | vs total gain: rho = {rt:+.2f} (p = {pt:.2g})')

# (c) Judge labels: original vs cleaned
print()
print(pd.crosstab(J.qwen4b_label_orig, J.qwen4b_label_clean).reindex(index=LABELS, columns=LABELS, fill_value=0))

## 5. Bonus: clean every resume. Does ranking get better?

Every resume in every job is cleaned, then each method ranks the candidates again. NDCG@10 per job, the same definition as Notebook 3b (gain 2^rel − 1, random tie-break). Paired permutation test, Holm across the four methods. The `orig` numbers should be close to Notebook 3b (E5 about 0.735, TF-IDF about 0.706).

In [ ]:
def dcg(rels):
    rels = np.asarray(rels, dtype=float)
    return np.sum((2 ** rels - 1) / np.log2(np.arange(2, len(rels) + 2)))
def ndcg_at_k(r, k):
    ideal = dcg(sorted(r, reverse=True)[:k])
    return dcg(r[:k]) / ideal if ideal > 0 else 0.0
def paired_permutation_p(a, b, n=10000, seed=SEED):
    rng = np.random.default_rng(seed); d = np.asarray(a) - np.asarray(b)
    signs = rng.choice([-1, 1], size=(n, len(d)))
    return float((np.abs((signs * d).mean(axis=1)) >= abs(d.mean())).mean())

rows = []
for col in CONTENT:
    per = {}
    for v, sc in (('orig', col), ('allclean', col + '_allclean')):
        rng = np.random.default_rng(SEED); vals = []
        for jid, grp in evald.groupby('job_id', sort=True):
            s = grp[sc].to_numpy(); rel = grp.relevance.to_numpy()
            vals.append(ndcg_at_k(rel[np.lexsort((rng.random(len(s)), -s))].tolist(), 10))
        per[v] = np.array(vals)
    rows.append({'method': col, 'jobs': len(per['orig']), 'NDCG@10 orig': round(per['orig'].mean(), 4),
                 'NDCG@10 all cleaned': round(per['allclean'].mean(), 4),
                 'diff': round((per['allclean'] - per['orig']).mean(), 4),
                 'p': paired_permutation_p(per['allclean'], per['orig'])})
ndcg = pd.DataFrame(rows)
ndcg['p (Holm)'] = holm(ndcg.p.to_numpy())
ndcg

## 6. Save

In [ ]:
main.to_csv(f'{RESULTS}/08b_steps.csv', index=False)
strict.to_csv(f'{RESULTS}/08b_steps_strict.csv', index=False)
ndcg.to_csv(f'{RESULTS}/08b_allclean_ndcg.csv', index=False)
keep = ['job_id', 'resume_id', 'pair_key', 'relevance'] + \
       [f'{c}_{v}' for c in CONTENT for v in ('orig', 'clean', 'plain', 'pol')] + \
       [f'{c}_rank_{v}' for c in CONTENT for v in ('orig', 'clean', 'plain', 'pol')]
target[keep].to_parquet(f'{RESULTS}/08b_pair_scores.parquet', index=False)
J[['pair_key', 'resume_id', 'relevance'] + [f'qwen4b_{v}' for v in ('orig', 'clean', 'pol')] +
  [f'qwen4b_label_{v}' for v in ('orig', 'clean', 'pol')]].to_parquet(f'{RESULTS}/08b_judge_scores.parquet', index=False)
rw[['resume_id', 'messiness', 'strict']].to_csv(f'{RESULTS}/08b_messiness.csv', index=False)
print('saved. Next: download this notebook (File > Download > .ipynb).')